In [35]:
import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag



spell = Speller(lang='en') # tool to auto-fix spelling mistakes
stop_words = set(stopwords.words('english')) # list of common words removed as it has no meaning
lemmatizer = WordNetLemmatizer() # tool to reduce words to their base form

# converts short forms to full meaning
slang_dict = { 
    "tbh": "to be honest", "omg": "oh my god", "lol": "laugh out loud", 
    "idk": "I don't know", "ig": "I guess", "ik": "I know", "esp": "especially", "coz": "because", "cuz":"because"
}

# fixing common contractions
contractions_dict = {
    "wasnt": "was not", "isnt": "is not", "cant": "cannot",
    "dont": "do not", "doesnt": "does not",
    "didnt": "did not"
}

malay_dict = {
    "dobi": "laundry", "sapu": "sweep", "bakul": "basket", "ye":""
}

# fixing common spelling mistakes found in the dataset
spelling_dict = {
    "tha": "the", "dri": "dry", "easi":"easy", "caus":"because", "mainten":"maintenance", "separ":"seperate",
    "chang":"change", "hygen":"hygiene", "facil":"facility","appreci":"appreciation","bfr":"before",
    "nahh":"no", "environ":"environment", "conveni":"convenience", "manag":"manage", "improv":"improve", 
    "doe":"does", "gener":"generally", "locat":"location", "everytim":"everytime"
}

# fix weird encoding characters that sometimes appear in scraped text (might be due to importing emoji)
def fix_encoding(text):
    if not isinstance(text, str): return ""
    text = text.replace('ÃƒÂ¢Ã¢Â‚Â¬Ã¢Â„Â¢', "'").replace('â€™', "'").replace('â\x80\x99', "'")
    return text.encode('utf-8', 'ignore').decode('utf-8')

def replace_slang(text):
    for word, full in slang_dict.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', full, text, flags=re.IGNORECASE) #finds the word existing by itself rather than existing within a word (eg 'ig' in 'running') ignoring the casing
    return text

def translate_malay(text):
    for word, meaning in malay_dict.items():
        text = re.sub(r'\b' + word + r'\b', meaning, text)
    return text

def remove_punctuation(text):
    # Keep spaces between words when punctuation is removed
    return text.translate(str.maketrans('', '', string.punctuation))

def replace_contractions(text):
    for word, full in contractions_dict.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', full, text, flags=re.IGNORECASE) 
    return text
    
def fix_common_spelling(text):
    for wrong, correct in spelling_dict.items():
        text = re.sub(r'\b' + wrong + r'\b', correct, text)
    return text


def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# autocorrect spelling using library
def correct_spelling(text):
    return spell(text)

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word.lower() not in stop_words])


# convert POS tag to WordNet format for lemmatization
# tells the lemmatizer if the word is noun/verb/adjective/etc (to standardize)
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'): return wordnet.ADJ
    elif nltk_tag.startswith('V'): return wordnet.VERB
    elif nltk_tag.startswith('N'): return wordnet.NOUN
    elif nltk_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

# convert words into their root/base form
def lemmatize_text(text):
    if not isinstance(text, str): return ""
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags])


# runs all cleaning steps one by one
def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == "": return []    #If this piece of data isn't a string, or if it’s just an empty spaces, just return an empty list
    
    text = fix_encoding(text)         # Step 0: Fix 'ÃƒÂ¢'
    text = text.lower()               # Step 1: Lowercase
    text = replace_slang(text)        # Step 2: Slang
    text = translate_malay(text)      # Step 3: translates malay to english
    text = remove_punctuation(text)   # Step 4: Punctuation
    text = replace_contractions(text) # Step 5: Contractions
    text = fix_common_spelling(text)  # Step 6: fixes common spelling errors not detected by autocorrect ('spell')
    text = remove_numbers(text)       # Step 7: Numbers
    text = correct_spelling(text)     # Step 8: Spelling
    text = remove_stopwords(text)     # Step 9: Stopwords
    text = lemmatize_text(text)       # Step 10: Lemmatization
    
    return word_tokenize(text)        # Step 11: Final Tokenization


#_____________________________________________________________________________________________________
#FILE: L.csv
try:
    df = pd.read_csv("L.csv", encoding='utf-8-sig') # try reading file with UTF-8 first
                                                    # if it fails, fallback to latin1 encoding
except UnicodeDecodeError:
    df = pd.read_csv("L.csv", encoding='latin1')

target_col = df.columns[0] #the first column containing the text reviews

df["processed"] = df[target_col].astype(str).apply(preprocess_text) # applies preprocessing to every row
df.to_csv("Processed_L.csv", index=False)
print(f"Processed {len(df)} rows.")
print(df[[target_col, "processed"]].head())

#___________________________________________________________________________________________________
#FILE: M.csv
try:
    df = pd.read_csv("M.csv", encoding='utf-8-sig')
except UnicodeDecodeError:
    df = pd.read_csv("M.csv", encoding='latin1')

target_col = df.columns[0]

df["processed"] = df[target_col].astype(str).apply(preprocess_text)
df.to_csv("Processed_M.csv", index=False)
print(f"Processed {len(df)} rows.")
print(df[[target_col, "processed"]].head())

#___________________________________________________________________________________________________
#FILE: N.csv
try:
    df = pd.read_csv("N.csv", encoding='utf-8-sig')
except UnicodeDecodeError:
    df = pd.read_csv("N.csv", encoding='latin1')

target_col = df.columns[0]

df["processed"] = df[target_col].astype(str).apply(preprocess_text)
df.to_csv("Processed_N.csv", index=False)
print(f"Processed {len(df)} rows.")
print(df[[target_col, "processed"]].head())


#___________________________________________________________________________________________________
#FILE: O.csv
try:
    df = pd.read_csv("O.csv", encoding='utf-8-sig')
except UnicodeDecodeError:
    df = pd.read_csv("O.csv", encoding='latin1')

target_col = df.columns[0]

df["processed"] = df[target_col].astype(str).apply(preprocess_text)

df.to_csv("Processed_O.csv", index=False)
print(f"Processed {len(df)} rows.")
print(df[[target_col, "processed"]].head())

#___________________________________________________________________________________________________

Processed 61 rows.
  Describe your overall experience using the campus laundry service    \
0          usual poor experi machin environ unhygien                    
1           overal s good just minor discrep disrupt                    
2                                       okay satisfi                    
3                               just okay best worst                    
4  doesnt dri cloth s bit expens includ soap pref...                    

                                           processed  
0  [usual, poor, expert, machine, environment, un...  
1          [overall, good, minor, discrete, disrupt]  
2                                    [okay, satisfy]  
3                                [okay, best, worst]  
4  [dry, cloth, bit, expense, include, soap, pref...  
Processed 63 rows.
  What problems have you experienced when using the laundry service?    \
0  \nWrong timer, dryer not drying properly,\nUnc...                     
1  Sometimes the payment is made, however the